# Coursework 2

#### [ ] I would like to volunteer to give the presentation.
Change [ ] to [x] to tick it.

In [1]:
using JLD2, FileIO

## 1. Your Data 

### Data Type Declaration

The data in the provided data file are stored using the data types defined below. These type definitions are required to load the data correctly.

**Do not modify these definitions.**

In [2]:
# Data type for Q9_1
#########################
#  Primitive Root
#########################
mutable struct QPrimRoot
    p :: BigInt 
    q :: Int
    a :: BigInt 
    b :: BigInt 
    a_is_prim :: Union{Missing,Bool}
    x :: Union{Missing,BigInt} 
    prim_list :: Union{Missing,Vector{Int}}
end

# Data type for Q9_2
#########################
# Discrete Log
#########################
mutable struct QDisLog
    p :: Int 
    b1 :: Int
    y_k :: Vector{Int}
    x_k :: Union{Missing,Vector{Vector{Int}}}
    # each x[k], k=1,2,3 is of type Vector{Int}
end

# Data type for Q10_1
#########################
# RSA Encryption
#########################
mutable struct QRSAEncrypt
    p :: BigInt 
    q :: BigInt 
    n :: Union{Missing, BigInt}
    e :: Union{Missing, BigInt}
    bat_ctext :: Union{Missing, BigInt}
    cat_ctext :: Union{Missing, BigInt}
    hat_ctext :: Union{Missing, BigInt}
end

# Data type for Q10_2
############################
# Cryptography: Discrete Log
############################
mutable struct QCryptDisLog
    p :: BigInt
    g :: Union{Missing, BigInt}
    A_pubkey :: Union{Missing, BigInt}
    B_pubkey :: Union{Missing, BigInt}
    C_pubkey :: Union{Missing, BigInt}
    DH_A_B :: Union{Missing, BigInt}
    DH_B_C :: Union{Missing, BigInt}
    DH_A_C :: Union{Missing, BigInt} 
    B_k :: BigInt
    B2A_ctext :: Union{Missing, Vector{BigInt}}
    A_signed :: Union{Missing, Tuple{String, BigInt, BigInt}}
    B_signed :: Tuple{String, BigInt, BigInt}
    B_signed_verified :: Union{Missing, Bool}
    C_signed :: Tuple{String, BigInt, BigInt}
    C_signed_verified :: Union{Missing, Bool}
end

# Data type for Q10_3
#########################
# Shamir Secret Sharing
#########################
mutable struct QShamir
    p1 :: Int 
    xk_1 :: Vector{Int}
    yk_1 :: Vector{Int}
    yklk :: Union{Missing,Vector{Vector{Int}}}
    fx :: Union{Missing,Vector{Int}}
    p2 :: Int
    xk_2 :: Vector{Int}
    yk_2 :: Vector{Int}
    k_incorrect :: Union{Missing,Int}
    shared_secret :: Union{Missing,Int}
end

# Data type for Q11_1
####################################
# Congruence Inverse for Polynomials
####################################
mutable struct QCongInvPoly
    ax :: Vector{Int}
    bx :: Vector{Int}
    mx :: Vector{Int}
    px :: Union{Missing,Vector{Vector{Int}}}
end

# Data type for Q11_2
##########################################
# Primitive Elements for Polynomial Fields
##########################################
mutable struct QPrimElPolyF 
    # 11.2.1
    p :: Int64                              # Given prime p
    f :: Vector{Int64}                      # Given coefficients of f(x)
    q :: Vector{Int64}                      # Given coefficients of q(x)
    q_is_primitive :: Union{Missing,Bool}    # Answer: is q(x) primitive?
    # 11.2.2
    f1 :: Vector{Int64}                     # Given coefficients of f₁(x)
    primitive_elements_f1 :: Union{Missing,Vector{Vector{Int64}}}
    # 11.2.3
    f2 :: Vector{Int64}                     # Given coefficients of f₂(x)
    primitive_elements_f2 :: Union{Missing,Vector{Vector{Int64}}}
end


### Load Your Data

In [4]:
fdata_name = "CCT-CW2-06124635.jld2"

ans_Q9_1 = load(fdata_name, "ans_Q9_1")
ans_Q9_2 = load(fdata_name, "ans_Q9_2")
ans_Q10_1 = load(fdata_name, "ans_Q10_1")
ans_Q10_2 = load(fdata_name, "ans_Q10_2")
ans_Q10_3 = load(fdata_name, "ans_Q10_3")
ans_Q11_1 = load(fdata_name, "ans_Q11_1")
ans_Q11_2 = load(fdata_name, "ans_Q11_2");

## 2. Your Answers

### 9.1 Primitive Root

In [6]:
const SMALL_BASES = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41)

const BASES_64 = (2, 325, 9375, 28178, 450775, 9780504, 1795265022)

# Primality test of Miller-Rabin.
function is_prime_number(n::T) where {T <: Integer}
    #Input validation: prime numbers are greater than 1
    if (n <= 1)
        return false
    end

    n = BigInt(n)
    
    #Check if p is divided by small primes in BASES
    for p in SMALL_BASES
        if n == p
            return true
        end
        if n % p == 0 #n is divisible by p
            return false
        end
    end

    # Handle even numbers
    if iseven(n)
        return false
    end

    #Decompose n-1 = d*2^s, with d odd
    d = n - 1
    s = 0
    while iseven(d)
        d = div(d,2)
        s += 1
    end

    #Miller-Rabin test for each base a
    for a in BASES_64
        if a % n == 0
            continue
        end
        
        x = powermod(BigInt(a), d, n)

        #Check the two first conditions
        if (x == 1 || x == n - 1)
            continue
        end
        
        composite = true

        #repeated squared modulo n
        for _ in 1:(s - 1)
            x = mod(x * x, n)
            if x == n - 1
                composite = false
                break
            end
        end
        
        if composite
            return false
        end
    end
    return true
end


f(x,c,n) = mod(powermod(x,2,n) + c, n)
#This function computes Pollard's Rho algorithm for finding a non-trivial factor
function pollard_rho(n::T) where {T <: Integer}
    #Input validation: prime factorisation is only definite for positive integers
    if (n <= 1)
        throw(ArgumentError("The input must be greater than 1"))
    end

    if mod(n,2) == 0
        return T(2)
    end
    
    N = BigInt(n)
    while true
        c = rand(BigInt(1):N-1)
        x = rand(BigInt(2):N-1)
        y = x
        d = 1
            
        while d == 1
            x = f(x,c,N)
            y = f(f(y,c,N),c,N)
            d = gcd(abs(x-y), N)
        end

        if 1 < d < n
            return T(d)
        end
    end
end

#This function returns non-banal factors of n
function prime_factors(n::T) where {T <: Integer}
    #Input validation: n must be greater than 1
    if (n <= 1)
        throw(ArgumentError("The input must be greater than 1"))
    end

    factors= T[]

    while n>1 && !is_prime_number(n)
        factor = pollard_rho(n)
        ret = prime_factors(factor)

        for i in ret
            while n%i == 0
                n = div(n,i)
            end
        end

        append!(factors, ret)
    end

    if n>1 #n is a prime number
        push!(factors, n)
    end

    return unique(factors)
end


#--------------------------   1   --------------------------
#This function checks primitive root condition directly, given (or computing) prime factors of p-1
function check_primitive_root(a::T1, p::T2, factors::Vector{T2} = T2[]) where {T1 <: Integer,T2 <: Integer}
    #edge case
    if p == 2
        return a == 1
    end
    
    pMinus = p - T2(1)
    
    if isempty(factors)
        factors = prime_factors(pMinus)
    end

    #Test: a is a primitive root <=> a^((p-1)/q)!=1 (mod p) for every distinct prime factor q of p-1 
    for q in factors
        if (powermod(a, div(pMinus, q), p) == T2(1))
            return false 
        end
    end 

    return true
end
    
#This function checks if a is a primitive root modulo p. 
function is_primitive_root(a::T1, p::T2) where {T1 <: Integer,T2 <: Integer}
    #Input validation: p must be a prime number
    if !is_prime_number(p)
        return false
    end
    
    pMinus = p - T2(1)
    
    if !(1 <= a <= pMinus)
        return false #By definition, if the condition fails a is not a primitive root
    end

    return check_primitive_root(a, p)
end

    ans_Q9_1.a_is_prim = is_primitive_root(ans_Q9_1.a, ans_Q9_1.p)


#--------------------------   2   --------------------------

#This function takes a number b and a prime number p, finds the first x>=b such that x is a primitive root of p and returns x mod p
function next_primitive_root(b::T1, p::T2) where {T1 <: Integer,T2 <: Integer}
    if p == 2
        return T2(1)
    end

    pMinus = p - T2(1)
    factors = prime_factors(pMinus)

    x = BigInt(b)
    while true #since p is prime, there always exists a primitive root
        candidate = mod(x,p)
        if (candidate != 0) && check_primitive_root(candidate, p, factors)
            return candidate
        end

        x += 1
        
    end
end

    ans_Q9_1.x = next_primitive_root(ans_Q9_1.b, ans_Q9_1.p)

#--------------------------   3   --------------------------

#This function returns all primitive roots of the prime number p
function all_primitive_roots(p::T) where {T <: Integer}
    if !is_prime_number(p)
        throw(ArgumentError("p must be prime"))
    end

    if p == 2
        return T[1]
    end
    
    roots = T[]

    pMinus = p - T(1)
    factors = prime_factors(pMinus)
    
    for candidate in T(1):pMinus
        if check_primitive_root(candidate, p, factors)
            push!(roots, candidate)
        end
    end
    return roots
end

    ans_Q9_1.prim_list = all_primitive_roots(ans_Q9_1.q)

    ans_Q9_1




QPrimRoot(3394189843926643, 479, 1131396614642214, 678837968785328, false, 678837968785328, [13, 17, 19, 26, 29, 31, 34, 37, 38, 39  …  468, 469, 470, 471, 472, 473, 474, 475, 476, 477])

### 9.2 Discrete Log

In [8]:

#This function takes a a prime number p, a primitive root of p b and an integer y!=0. 
#It returns the discrete log of a nonzero residue y with respect to b (mod p)
function discrete_log(p::T,b::T,y::T) where {T <: Integer}
    #Input validation
    if !is_prime_number(p)
        throw(ArgumentError("p must be prime"))
    end
    if !is_primitive_root(b, p)
        throw(ArgumentError("b must be a primitive root of p"))
    end
    
    y = mod(y,p)
    if iszero(y)
        throw(ArgumentError("y must be different than 0"))
    end

    
    if y == T(1) #base case: b^0 = 1 (mod p)
        return [T(0)]
    else
        current = T(1)

        #Since b is a primitive root of p the discrete logarithm 1 <= x <= p-2 
        pMinusTwo = p - T(2)
        for x in T(1):pMinusTwo
            current = mod(BigInt(b) * current, p)

            if (current == y)
                return T[x]
            end
        end
    end

    error("Discrete logarithm not found")
end

    dim = size(ans_Q9_2.y_k)[1]
    ans_Q9_2.x_k = Vector{Vector{Int}}(undef, dim)
    for i in 1: dim
        ans_Q9_2.x_k[i] = discrete_log(ans_Q9_2.p,ans_Q9_2.b1,ans_Q9_2.y_k[i])
    end



#Verification
    for i in 1:dim
        res = powermod(ans_Q9_2.b1, ans_Q9_2.x_k[i][1], ans_Q9_2.p)
        println("res: ", res)
        println(res == mod(ans_Q9_2.y_k[i], ans_Q9_2.p))
    end




res: 4
true
res: 5
true
res: 6
true


### 10.1 RSA Encryption

In [10]:
ans_Q10_1 = load(fdata_name, "ans_Q10_1")

#--------------------------   1   --------------------------

#This function returns the first number m>=e which is co-prime with n
function first_coprime(n::T1, e::T2) where {T1 <: Integer, T2 <: Integer}
    #Input validation
    if (n < 1)
        throw(ArgumentError("n must be greater or equal than 1"))
    end
    
    m = e 
    while true
        if (gcd(n,m) == 1)
            return m
        else
            m += 1
        end
    end
end


#This function takes the two prime numbers p and q and computes the public and the private keys
#check_primes indicates if you have to check if p and q are primes
function key_generation(p::T1, q::T2, e_start::T3, check_primes::Bool = false) where {T1 <: Integer, T2 <: Integer, T3 <: Integer}
    #Input validation: p and q must be primes
    if check_primes
        if !is_prime_number(p) || !is_prime_number(q)
            throw(ArgumentError("p and q must be prime"))
        end
    end

    #Input validation: p!=q
    if p == q
        throw(ArgumentError("p and q must be different"))
    end

    # Compute safely
    n_big = BigInt(p) * BigInt(q)
    t_big = (BigInt(p) - 1) * (BigInt(q) - 1)

    # Use Int64 if safe
    if (n_big <= typemax(Int64)) && (t_big <= typemax(Int64)) && (typemin(Int64) <= e_start <= typemax(Int64))
        n = Int64(n_big)
        t = Int64(t_big)
        e = first_coprime(t, Int64(e_start))
    else
        n = n_big
        t = t_big
        e = first_coprime(t, BigInt(e_start))
    end



    return n,e
end

const e_start = 305407503811

    ans_Q10_1.n, ans_Q10_1.e = key_generation(ans_Q10_1.p, ans_Q10_1.q, e_start)



#--------------------------   2   --------------------------

# This function converts a plaintext string into an integer for RSA encryption
function plaintext_to_integer(ptext::String)
    # Keep only lowercase letters
    ptext = filter(c -> 'a' <= c <= 'z', ptext)

    if isempty(ptext)
        throw(ArgumentError("Plaintext must contain at least one lowercase letter"))
    end

    m = BigInt(0)

    for c in ptext
        value = Int(c) - Int('a') + 1
        m = m * 100 + value
    end

    return m
end

#This function RSA encrypt the ptext
function RSA_encrypt(ptext::String, n::T1, e::T2) where {T1 <: Integer, T2 <: Integer}
    if e <= 0 || n <= 1
        throw(ArgumentError("Invalid RSA public key"))
    end
    
    m = plaintext_to_integer(ptext)

    if m >= n
        throw(ArgumentError("Plaintext must be smaller than the RSA modulus n"))
    end

    return powermod(m, e, n)
end

    ans_Q10_1.bat_ctext = RSA_encrypt("bat", ans_Q10_1.n, ans_Q10_1.e)
    ans_Q10_1.cat_ctext = RSA_encrypt("cat", ans_Q10_1.n, ans_Q10_1.e)
    ans_Q10_1.hat_ctext = RSA_encrypt("hat", ans_Q10_1.n, ans_Q10_1.e)

    println("bat cyphertext: ", ans_Q10_1.bat_ctext)
    println("cat cyphertext: ", ans_Q10_1.cat_ctext)
    println("hat cyphertext: ", ans_Q10_1.hat_ctext)

#=
Although the plaintexts "bat", "cat", and "hat" differ by only one letter, their corresponding RSA ciphertexts are significantly 
different. This illustrates the nonlinear effect of modular exponentiation in RSA encryption: a small change in the plaintext can 
result in a large numerical change in the ciphertext.
=#



bat cyphertext: 413445163278263083351418897918161309134334030396
cat cyphertext: 160874559896915332026619838725288127116978626772
hat cyphertext: 22953742899432169684833268552063050866942334809


### 10.2 Cryptography: Discrete Log

In [13]:
#--------------------------   1   --------------------------

    ans_Q10_2.g = next_primitive_root(e_start, ans_Q10_2.p)



#--------------------------   2   --------------------------
#This function converts a plaintext string into a private key
#and computes the corresponding public key using modular exponentiation
function public_private_keys(p::T1, g::T2, ptext::String) where{T1 <: Integer, T2 <: Integer}
    a = plaintext_to_integer(ptext)
    A = powermod(g,a,p)
    return a, A
end


    a_privkey, ans_Q10_2.A_pubkey = public_private_keys(ans_Q10_2.p, ans_Q10_2.g, "apple")
    b_privkey, ans_Q10_2.B_pubkey = public_private_keys(ans_Q10_2.p, ans_Q10_2.g, "banana")
    c_privkey, ans_Q10_2.C_pubkey = public_private_keys(ans_Q10_2.p, ans_Q10_2.g, "cherry")


#--------------------------   3   --------------------------

#This function computes the Diffie-Hellman shared secret
#using one participant's private key a and the other participant's public key B
function DH_exchange(a::T1, B::T2, p::T3) where{T1 <: Integer, T2 <: Integer, T3 <: Integer}
    return powermod(B, a, p)
end

    ans_Q10_2.DH_A_B = DH_exchange(a_privkey, ans_Q10_2.B_pubkey, ans_Q10_2.p)
    ans_Q10_2.DH_B_C = DH_exchange(b_privkey, ans_Q10_2.C_pubkey, ans_Q10_2.p)
    ans_Q10_2.DH_A_C = DH_exchange(a_privkey, ans_Q10_2.C_pubkey, ans_Q10_2.p)
    
        
#--------------------------   4   --------------------------

#This function encrypts the plaintext using ElGamal encryption
#with the recipient's public key (p,g,a) and the random integer k
#It returns the ciphertext pair [c1,c2]
function ElGamal_encrypt(p::T1, g::T2, a::T3, k::T4, ptext::String) where{T1 <: Integer, T2 <: Integer, T3 <: Integer, T4 <: Integer}
    m = plaintext_to_integer(ptext)
    c1 = powermod(g, k, p)
    c2 = powermod(a, k, p)
    c2 = mod(mod(m,p) * c2, p)
    return [c1, c2]
end

    @assert plaintext_to_integer("nice") < ans_Q10_2.p

    ans_Q10_2.B2A_ctext = ElGamal_encrypt(ans_Q10_2.p, ans_Q10_2.g, ans_Q10_2.A_pubkey, ans_Q10_2.B_k, "nice")

#--------------------------   5   --------------------------

#This function computes k>=start such that gcd(k,p)==1
function next_coprime(p::T1, start::T2) where{T1 <: Integer, T2 <: Integer}
    x = BigInt(start)
    
    while true
        if gcd(p, x) == 1
            return x
        end
        
        x+=1
    end
end


#This function computes the ElGamal digital signature
function ElGamal_digital_signature(p::T1, g::T2, a::T3, k::T4, ptext::String) where{T1 <: Integer, T2 <: Integer, T3 <: Integer, T4 <: Integer}
    pMinus = BigInt(p) - 1
    
    #Input validation: p-1 and k must be coprime
    if gcd(pMinus,k)!= 1
            throw(ArgumentError("p-1 and k must be coprime"))
    end

    m =  plaintext_to_integer(ptext)
        
        
    r = powermod(g, k, p)
    
    ret = invmod(k, pMinus)
    s = mod(ret * mod(m - BigInt(a) * r, pMinus), pMinus)

    return ptext, r, s
    
end

    ans_Q10_2.A_signed = ElGamal_digital_signature(ans_Q10_2.p, ans_Q10_2.g, a_privkey, next_coprime(ans_Q10_2.p - 1,5000), "seven")
    

#--------------------------   6   --------------------------
#This function verifies an ElGamal digital signature
function check_digital_signature(signature::Tuple{String, BigInt, BigInt}, p::BigInt, g::BigInt, a::BigInt)
    ptext, r, s = signature

    #Validate signature components
    if !(1 <= r < p && 0 <= s < p-1)
        return false
    end
    
    res = powermod(a, r, p) * powermod(r, s, p)
    v1 = mod(res, p)

    m = plaintext_to_integer(ptext)
    v2 = powermod(g, m, p)

    return v1 == v2
end
    
    ans_Q10_2.B_signed_verified = check_digital_signature(ans_Q10_2.B_signed, ans_Q10_2.p, ans_Q10_2.g, ans_Q10_2.B_pubkey)
    

#--------------------------   7   --------------------------

    ans_Q10_2.C_signed_verified = check_digital_signature(ans_Q10_2.C_signed, ans_Q10_2.p, ans_Q10_2.g, ans_Q10_2.C_pubkey)



true

### 10.3  Shamir Secret Sharing

In [15]:

#--------------------------   1   --------------------------

#This function multiplies a polynomial by x by shifting its coefficients one position towards higher powers
function shift_vector(x::Vector{T}) where{T <: Integer}
    dim = length(x)
    shift_x = zeros(BigInt, dim )
    for i in dim:-1:2
        shift_x[i] = x[i-1]
    end
    
    shift_x[1] = 0
    
    return shift_x
end

#This function computes the coefficients of the k-th Lagrange basis polynomial l_k(x) modulo p
function lk_x(x::Vector{T1}, k::Int, p::T2) where{T1 <: Integer, T2 <: Integer}
    dim = length(x)

    #Input validation: 1 <= k <= dim
    if !(1 <= k <= dim)
        throw(ArgumentError("non valid k"))
    end

    #Input validation: all denominators must be invertible modulo p
    for i in 1:dim
        for j in (i+1):dim
            if gcd(BigInt(x[i])-x[j],p) != 1
                throw(ArgumentError("It is not possible to compute the algorithm"))
            end
        end
    end
    
    sol = BigInt(1)
    x_k = BigInt(x[k])

    #Compute the product of the denominator factors (x_k - x_i) modulo p
    for i in 1:dim
        if i!=k
            sol = mod(sol * (x_k - x[i]), p)
        end
    end

    #Compute the modular inverse of the denominator
    sol = invmod(sol, p)

    #Initialize the numerator polynomial to 1
    #Coefficients are stored in ascending order of powers
    pol = zeros(BigInt, dim)
    pol[1] = 1

    #Compute the numerator polynomial as the product of (x - x_i)
    for i in 1:dim
        if i != k
            shifted_pol = shift_vector(pol)

            #Multiply the current polynomial by (x - x_i) modulo p
            pol = mod.(shifted_pol .- pol * x[i], p)
        end
    end

    #Multiply all coefficients by the inverse of the denominator
    pol = mod.(pol .* sol, p)

    return pol
end

    
#This function computes the Lagrange interpolation polynomial modulo p
function lagrange_interpolation(p::T1, x::Vector{T2}, y::Vector{T3}) where{T1 <: Integer, T2 <: Integer, T3 <: Integer}
    dim = length(x)

    #Input validation: p must be prime
    if !is_prime_number(p)
        throw(ArgumentError("p must be prime"))
    end
    
    #Input validation: x and y must have the same length
    if dim!=length(y)
        throw(ArgumentError("x and y must be of the same size"))
    end

    #Input validation: at least one point is required
    if dim == 0
        throw(ArgumentError("At least one point is required"))
    end

    #Compute and store the coefficients of all y_k*l_k(x)
    #Each row contains the coefficients of one polynomial
    yklk = Vector{Vector{T1}}(undef, dim)
    
    for k in 1 : dim
        y_k = mod(BigInt(y[k]), p)
        yklk[k] = mod.(y_k * lk_x(x, k, p), p)
    end

    #Reconstruct the interpolation polynomial by summing all y_k*l_k(x)
    f = copy(yklk[1])
    for i in 2:dim
        f = mod.(BigInt.(f) .+ yklk[i], p)
    end
         
    
    return yklk, f
end

    ans_Q10_3.yklk, ans_Q10_3.fx = lagrange_interpolation(ans_Q10_3.p1, ans_Q10_3.xk_1, ans_Q10_3.yk_1)


#--------------------------   2   --------------------------

# Computes f(x) mod p, with coeff[1] as the constant term.
function f(coeff::Vector{T1}, x::T2, p::T3) where {T1 <: Integer, T2 <: Integer, T3 <: Integer}
    dim = length(coeff)
    
    res = mod(BigInt(coeff[1]), p)
    x_i = BigInt(1)

    for i in 2:dim
        x_i = mod(x_i * x, p)
        res = mod(res + coeff[i] * x_i, p)
    end

    return res
end



function incorrect_sharing(p::T1, t::T2, x::Vector{T3}, y::Vector{T3}) where {T1 <: Integer, T2 <: Integer, T3 <: Integer}
    dim = length(x)

    if dim != length(y)
        throw(ArgumentError("x and y must have the same length"))
    end

    if !(1 < t <= dim - 2)
        throw(ArgumentError("t must be greater than 1 and at most length(x) - 2"))
    end

    if p <= dim || !is_prime_number(p)
        throw(ArgumentError("p must be prime and greater than length(x)"))
    end

    if length(unique(mod.(x, p))) != dim
        throw(ArgumentError("The x coordinates must be distinct modulo p"))
    end
    
    # Interpolate using the first t shares.
    _, pol = lagrange_interpolation(p, x[1:t], y[1:t])
    count = 0
    sol = 0

    for i in (t+1):dim
        if mod(y[i], p) != f(pol, x[i], p)
            count += 1
            sol = i
        end
    end

    if count == 1
        return sol
    elseif count == 0
        return nothing
    end
    
    # Replace each of the first t shares with share t+1, one at a time.
    for i in 1:t
        indices = [j for j in 1:(t+1) if j != i]
        _, pol = lagrange_interpolation(p, x[indices], y[indices])
        count = 0
        sol = 0

        if mod(y[i], p) != f(pol, x[i], p)
            count += 1
            sol = i
        end

        for j in (t+2):dim
            if mod(y[j], p) != f(pol, x[j], p)
                count += 1
                sol = j
            end
        end

        if count == 1
            return sol
        end
    end

    return nothing
end

    ans_Q10_3.k_incorrect = incorrect_sharing(ans_Q10_3.p2, 4, ans_Q10_3.xk_2, ans_Q10_3.yk_2)

    indices = [i for i in eachindex(ans_Q10_3.xk_2) if i != ans_Q10_3.k_incorrect][1:4]
    _, pol = lagrange_interpolation(ans_Q10_3.p2, ans_Q10_3.xk_2[indices], ans_Q10_3.yk_2[indices])
    ans_Q10_3.shared_secret = pol[1]



258632270033

### 11.1 Congruence Inverse for Polynomials

In [17]:
using AbstractAlgebra, Nemo


# This function returns the nonzero solutions of a*p ≡ b (mod m) over a prime field, with degree less than degree(m).
function modularDivPolynomial(a::T, b::T, m::T) where {T <: AbstractAlgebra.PolyRingElem}
    R = parent(a)   # The polynomial ring containing a, b, and m
    F = base_ring(R)
    p = Int(Nemo.characteristic(F))

    res = T[]
    d = gcd(a, m)

    # A solution exists only if d divides b.
    # If it does, b1 is the exact quotient b/d.
    ok, b1 = Nemo.divides(b, d)

    if !ok
        return res
    end

    # Reduce the congruence to a1*p ≡ b1 (mod m1)
    a1 = divexact(a, d)
    m1 = divexact(m, d)

    # Find the base solution x1 modulo m1.
    if degree(m1) == 0
        x1 = zero(R)   # The remainder modulo 1 is always zero.
    else
        y = invmod(a1, m1)
        x1 = mod(y * b1, m1)
    end

    # For k = 0, the solution x1 + k*m1 reduces to x1.
    if !iszero(x1)
        push!(res, x1)
    end

    if degree(d) == 0
        return res
    end
    
    ranges = ntuple(_ -> 0:(p-1), degree(d))
    
    # Generate every k of degree less than degree(d).
    for c in Iterators.product(ranges...)
        k = R(collect(c))
        
        # Skip k = 0 because it was handled above.
        if !iszero(k)
            push!(res, x1 + k * m1)
        end
    end

    return res
end

    R, x = Nemo.polynomial_ring(Nemo.GF(3), "x")

    for q in ans_Q11_1
        sols = modularDivPolynomial(R(q.ax), R(q.bx), R(q.mx))

        q.px = Vector{Int}[[Int(Nemo.lift(Nemo.ZZ, Nemo.coeff(p, i))) for i in 0:Nemo.degree(p)] for p in sols]
    end

    ans_Q11_1



3-element Vector{QCongInvPoly}:
 QCongInvPoly([2, 0, 1, 1], [2, 1, 1], [2, 2, 2, 0, 2], [[0, 2, 2]])
 QCongInvPoly([1, 2, 1, 2, 2], [1, 1, 2], [2, 0, 0, 1, 0, 1], Vector{Int64}[])
 QCongInvPoly([2, 0, 2, 1, 1], [2, 1, 2, 1], [1, 1, 0, 2, 1, 1], [[0, 1, 0, 1], [2, 2, 1, 0, 1], [1, 0, 2, 2, 2]])

### 11.2 Primitive Elements for Polynomial Fields

In [19]:

#--------------------------   1   --------------------------

function check_primitive(a::T, f::T, N::Int, factors) where {T <: AbstractAlgebra.PolyRingElem}
    a = mod(a, f)

    # Zero is outside the multiplicative group.
    if iszero(a)
        return false
    end

    # Test a^(N/q) ≠ 1 (mod f) for each prime factor q of N.
    for q in factors
        if isone(powermod(a, div(N, q), f))
            return false
        end
    end

    return true
end
 

# Assumes a and f are in the same ring Fₚ[x] and f is irreducible.
function is_primitive(a::T, f::T) where {T <: AbstractAlgebra.PolyRingElem}
    F = Nemo.base_ring(parent(f))
    p = BigInt(Nemo.characteristic(F))
    N = Int(p^Nemo.degree(f) - 1)

    factors = [Int(q) for (q, _) in Nemo.factor(Nemo.ZZ(N))]
    
    return check_primitive(a, f, N, factors)
end

    R, x = Nemo.polynomial_ring(Nemo.GF(ans_Q11_2.p), "x")
    ans_Q11_2.q_is_primitive = is_primitive(R(ans_Q11_2.q), R(ans_Q11_2.f))

#--------------------------   2   --------------------------

# Return all primitive elements of F₂[x]/(f), assuming f is irreducible.
function all_primitive(f::T) where {T <: AbstractAlgebra.PolyRingElem}
    R = parent(f)
    n = Nemo.degree(f)
    N = Int(big(2)^n - 1)

    if (N == 1)
        factors = Int[]
    else
        factors = [Int(q) for (q, _) in Nemo.factor(Nemo.ZZ(N))]
    end

    roots = T[]
    ranges = ntuple(_ -> 0:1, n)

    for coeffs in Iterators.product(ranges...)
        candidate = R(collect(coeffs))
        if check_primitive(candidate, f, N, factors)
            push!(roots, candidate)
        end
    end

    return roots
end

    R2, x2 = Nemo.polynomial_ring(Nemo.GF(2), "x")

    f1 = R2(ans_Q11_2.f1)
    roots = all_primitive(f1)

    ans_Q11_2.primitive_elements_f1 = Vector{Int64}[[Int64(Nemo.lift(Nemo.ZZ, Nemo.coeff(q, i))) for i in 0:Nemo.degree(q)]
    for q in roots]

#--------------------------   2   --------------------------


    f2 = R2(ans_Q11_2.f2)
    roots_f2 = all_primitive(f2)

    ans_Q11_2.primitive_elements_f2 = Vector{Int64}[[Int64(Nemo.lift(Nemo.ZZ, Nemo.coeff(q, i))) for i in 0:Nemo.degree(q)]
        for q in roots_f2]



36-element Vector{Vector{Int64}}:
 [1, 1]
 [1, 0, 1]
 [1, 1, 1]
 [0, 1, 0, 1]
 [0, 0, 1, 1]
 [0, 1, 1, 1]
 [1, 0, 0, 0, 1]
 [1, 1, 0, 0, 1]
 [1, 0, 1, 0, 1]
 [0, 0, 0, 1, 1]
 [0, 1, 0, 1, 1]
 [0, 0, 1, 1, 1]
 [1, 0, 0, 0, 0, 1]
 ⋮
 [0, 1, 0, 0, 1, 1]
 [1, 1, 0, 0, 1, 1]
 [0, 0, 1, 0, 1, 1]
 [1, 0, 1, 0, 1, 1]
 [0, 1, 1, 0, 1, 1]
 [1, 1, 1, 0, 1, 1]
 [0, 0, 0, 1, 1, 1]
 [1, 0, 0, 1, 1, 1]
 [0, 1, 0, 1, 1, 1]
 [1, 1, 0, 1, 1, 1]
 [1, 0, 1, 1, 1, 1]
 [1, 1, 1, 1, 1, 1]

## 3. Save All Your Answers

Remember to save your answers into your data file!

In [20]:
    jldsave(fdata_name;
     ans_Q9_1, ans_Q9_2, 
     ans_Q10_1, ans_Q10_2, ans_Q10_3,
     ans_Q11_1, ans_Q11_2
 );